In [ ]:
from scipy.io import arff
import pandas as pd
import numpy as np

# 1. Încarcă fișierul
data, meta = arff.loadarff('php0iVrYT.arff')
df = pd.DataFrame(data)

# 2. Decodare format bytes
for col in df.select_dtypes([object]).columns:
    df[col] = df[col].str.decode('utf-8')

print("Datele au fost încărcate!")
print(df.head())

: 

In [ ]:
# --- FEATURE ENGINEERING AVANSAT ---

# 1. Indicatori de Ritm și Viteza
df['Loyalty_Rate'] = df['V2'] / (df['V4'] + 1)
df['Donation_Speed'] = df['V2'] / (df['V4'] + 1) # Cât de des donează per lună

# 2. Indicatori de Perioadă și Inactivitate
df['Inactivity_Ratio'] = df['V1'] / (df['V4'] + 1)
df['Active_Duration'] = df['V4'] - df['V1'] # Perioada efectivă în care a fost activ

# 3. Indicatori Statistici și de Comportament
df['Is_Frequent'] = (df['V2'] > df['V2'].mean()).astype(int)
df['Avg_Wait_Period'] = df['V4'] / df['V2'] # Media de luni între două donații

# 4. Scoruri Combinate (Interacțiuni)
df['Recency_Frequency_Score'] = df['V1'] * df['V2']
df['Loyalty_Score'] = (df['V2'] * 10) / (df['V1'] + 1) # Scor mare = donator fidel și recent

# 5. Curățare finală pentru noile coloane
df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

print(f"Succes! Dataset-ul are acum {df.shape[1]} coloane.")

In [ ]:
# Verificare valori lipsă inițiale
print("Missing values înainte:", df.isnull().sum().sum())

# REPARARE: Înlocuim 0 cu NaN unde 0 nu este biologic posibil
# (Ex: nu poți avea BloodPressure sau BMI egal cu 0)
cols_fix = ['V1', 'V2', 'V3', 'V4'] # Ajustează conform numelor coloanelor tale
df[cols_fix] = df[cols_fix].replace(0, np.nan)

# Umplem cu mediana pentru a nu pierde rânduri
df.fillna(df.median(numeric_only=True), inplace=True)

print("Date curățate. Statistici descriptive:")
display(df.describe())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('Class', axis=1)
y = df['Class']

# Adăugăm stratify=y pentru echilibru
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datele au fost împărțite și scalate corect.")

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 1. Curățare și Mapare inițială
if 'V3' in df.columns:
    df = df.drop('V3', axis=1)

# Mapăm clasele: 1 devine 0 (Nu a donat), 2 devine 1 (A donat)
df['Class'] = df['Class'].map({'1': 0, '2': 1})

# --- 2. FEATURE ENGINEERING: Generăm noile atribute ---
# Folosim +1 la numitor pentru a evita împărțirea la zero

df['Loyalty_Rate'] = df['V2'] / (df['V4'] + 1)
df['Inactivity_Ratio'] = df['V1'] / (df['V4'] + 1)
df['Is_Frequent'] = (df['V2'] > df['V2'].mean()).astype(int)
df['Recency_Frequency_Score'] = df['V1'] * df['V2']
df['Active_Duration'] = df['V4'] - df['V1']
df['Donation_Speed'] = df['V2'] / (df['V4'] + 1)
df['Avg_Wait_Period'] = df['V4'] / (df['V2'] + 0.1)
df['Loyalty_Score'] = (df['V2'] * 10) / (df['V1'] + 1)

# Curățăm eventuale valori infinite generate de calcule
df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

# 3. Definirea X și y (Acum X include noile coloane automat)
X = df.drop('Class', axis=1)
y = df['Class']

# 4. Split și Scalare (Esențială pentru noile atribute cu scale diferite)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 5. SMOTE pentru echilibrare pe setul complex
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train_s, y_train)

# 6. Model XGBoost (Am ajustat puțin parametrii pentru complexitatea nouă)
model = XGBClassifier(
    n_estimators=200,       # Am crescut puțin numărul de arbori
    max_depth=4,            # Puțin mai adânc pentru noile corelații
    learning_rate=0.05,     # Învățare mai fină
    gamma=0.3,              # Control mai strict al complexității
    subsample=0.8,
    colsample_bytree=0.8,   # Selectează submulțimi de atribute
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_res, y_res)
y_pred = model.predict(X_test_s)

# Rezultate
print(f"Acuratețe Nouă (Dataset Complex): {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

In [ ]:
# ==========================================
# EVALUARE FINALĂ (După Feature Engineering)
# ==========================================

# 1. Calculăm acuratețea folosind noile predicții
# Asigură-te că y_test și y_pred provin din rularea cu noile coloane
accuracy = accuracy_score(y_test, y_pred)

print(f"Acuratețe Nouă (Dataset Complex): {accuracy * 100:.2f}%")
print("-" * 30)
print("Raport de Clasificare Detaliat:")
print(classification_report(y_test, y_pred))

# 2. Vizualizarea Matricei de Confuzie
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Calculăm matricea pe baza rezultatelor noi
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
# Am adăugat culori puțin mai intense pentru a evidenția diferențele
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu') 

plt.xlabel('Predicție (0=Nu a donat, 1=A donat)')
plt.ylabel('Realitate (0=Nu a donat, 1=A donat)')
plt.title('Matricea de Confuzie - Model cu Atribute Extinse')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
# Calculăm corelația pe tot setul de date (inclusiv noile atribute)
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matricea de Corelație a Atributelor')
plt.show()

In [ ]:
# Extragem importanța din modelul XGBoost deja antrenat
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Importanța Atributelor în Predicție')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Definim modelele
models = {
    "XGBoost": model, # Modelul pe care l-am făcut deja
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

results = []

for name, m in models.items():
    # Antrenăm (pe datele echilibrate SMOTE)
    m.fit(X_res, y_res)
    # Predicție
    y_p = m.predict(X_test_s)
    # Calculăm metrici
    acc = accuracy_score(y_test, y_p)
    f1 = f1_score(y_test, y_p)
    results.append({"Model": name, "Accuracy": acc, "F1-Score": f1})

# Afișăm tabelul comparativ
comparison_df = pd.DataFrame(results)
print(comparison_df)

In [ ]:
# 1. Identificăm și păstrăm doar atributele importante
# Eliminăm 'V4' și 'Inactivity_Ratio' conform analizei noastre
features_to_keep = [col for col in X.columns if col not in ['V4', 'Inactivity_Ratio']]

# Creăm noile seturi de date reduse
X_train_red = pd.DataFrame(X_train_s, columns=X.columns)[features_to_keep]
X_test_red = pd.DataFrame(X_test_s, columns=X.columns)[features_to_keep]

# Re-aplicăm SMOTE pe datele reduse pentru a păstra echilibrul
X_res_red, y_res_red = sm.fit_resample(X_train_red, y_train)

print(f"Număr atribute rămase: {len(features_to_keep)}")
print(f"Atribute utilizate: {features_to_keep}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Definim spațiul de căutare pentru parametri
param_dist = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'gamma': [0.1, 0.2, 0.3, 0.5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# Inițializăm căutarea
random_search = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42),
    param_distributions=param_dist,
    n_iter=15, # Testăm 15 combinații aleatorii
    cv=5,      # Folosim 5 fold-uri pentru fiecare combinație
    scoring='f1',
    random_state=42
)

random_search.fit(X_res_red, y_res_red)
best_model = random_search.best_estimator_

print("Cea mai bună combinație de parametri găsită:")
print(random_search.best_params_)

In [ ]:
from sklearn.model_selection import cross_val_score

# Calculăm scorul mediu pe 5 segmente diferite de date
cv_scores = cross_val_score(best_model, X_res_red, y_res_red, cv=5)

print(f"Scorurile fiecărui fold: {cv_scores}")
print(f"Media stabilității modelului (CV): {cv_scores.mean() * 100:.2f}% (+/- {cv_scores.std() * 100:.2f}%)")

In [ ]:
# Obținem probabilitățile de donare (între 0 și 1)
y_probs = best_model.predict_proba(X_test_red)[:, 1]

# Setăm un prag de 0.4 (coborâm pragul standard de 0.5 pentru a fi mai sensibili la donatori)
custom_threshold = 0.4
y_pred_final = (y_probs >= custom_threshold).astype(int)

print(f"Rezultate finale cu prag de {custom_threshold}:")
print("-" * 30)
print(f"Acuratețe Finală: {accuracy_score(y_test, y_pred_final) * 100:.2f}%")
print(classification_report(y_test, y_pred_final))

# Afișăm matricea de confuzie pentru noul model
cm_final = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(6,4))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Greens')
plt.title('Matrice de Confuzie Finală (Model Optimizat)')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# 1. Calculăm rata falșilor pozitivi (fpr) și rata adevăraților pozitivi (tpr)
fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

# 2. Generăm graficul
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curba ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Linia de referință (la întâmplare)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Rata Falșilor Pozitivi (FPR)')
plt.ylabel('Rata Adevăraților Pozitivi (TPR)')
plt.title('Curba ROC - Modelul Optimizat')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
import shap

# Explicăm modelul XGBoost (best_model din pasul anterior)
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_red)

# Vizualizare grafic de tip Summary (foarte bun pentru documentație)
shap.summary_plot(shap_values, X_test_red)

In [ ]:

import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Inițializare și antrenare
lgb_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)

# Predicții
lgb_preds = lgb_model.predict(X_test)
lgb_probs = lgb_model.predict_proba(X_test)[:, 1]

# Evaluare
print("--- Performanță LightGBM ---")
print(f"Acuratețe: {accuracy_score(y_test, lgb_preds):.2f}")
print(f"AUC: {roc_auc_score(y_test, lgb_probs):.2f}")
print(classification_report(y_test, lgb_preds))